### KBO 데이터 크롤링

### KBO 데이터 크롤링 코드 (KBO 공식 홈페이지)

In [21]:
# library
import pandas as pd
import requests
import xmltodict
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time

In [22]:
### 브라우저 열기 - chromeDriver
driver=webdriver.Chrome()

In [23]:
### 웹사이트
url="http://www.koreabaseball.com/Record/Player/HitterBasic/Basic1.aspx"
driver.get(url)

### 타자 데이터

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# ✅ 전체 데이터를 저장할 리스트
all_seasons = []

# ✅ 연도별 반복
for year in range(2018, 2026):
    driver = webdriver.Chrome()
    url = "https://www.koreabaseball.com/Record/Player/HitterBasic/Basic1.aspx"
    driver.get(url)

    # ✅ 연도/팀 선택
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlSeason_ddlSeason")).select_by_visible_text(str(year))
    time.sleep(1)

    # ✅ 팀명: 2021년부터는 SSG, 이전은 SK
    team_name = "키움" if year >= 2019 else "키움"
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlTeam_ddlTeam")).select_by_visible_text(team_name)
    time.sleep(2)


    # ✅ 페이지 수 확인 함수
    def get_total_pages():
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pages = soup.select('div.paging a')
        numbers = [int(p.text.strip()) for p in pages if p.text.strip().isdigit()]
        return max(numbers) if numbers else 1

    # ✅ 기본기록 크롤링
    basic_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not basic_all:
            basic_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        basic_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        basic_all.extend(basic_rows)
    df_basic = pd.DataFrame(basic_all, columns=basic_cols)

    # ✅ 세부기록 탭 클릭
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[2]/a[2]').click()
    time.sleep(2)

    # ✅ 세부기록 크롤링
    detail_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not detail_all:
            detail_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        detail_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        detail_all.extend(detail_rows)
    df_detail = pd.DataFrame(detail_all, columns=detail_cols)

    # ✅ 기본 + 세부기록 병합
    df_basic = df_basic.drop(columns=['순위', 'AVG'], errors='ignore')
    df_merged = pd.merge(df_detail, df_basic, on=['선수명', '팀명'], how='left')

    # ✅ PH-BA까지 정렬
    pre_stats = [
        '순위', '선수명', '팀명', 'AVG', 'G', 'PA', 'AB', 'R', 'H',
        '2B', '3B', 'HR', 'TB', 'RBI', 'SAC', 'SF', 'BB', 'IBB', 'HBP', 'SO', 'GDP',
        'SLG', 'OBP', 'OPS', 'MH', 'RISP', 'PH-BA'
    ]
    pre_stats = [col for col in pre_stats if col in df_merged.columns]
    df_final = df_merged[pre_stats]

    # ✅ XBH ~ GPA 탭 클릭
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[1]/ul/li[2]/a').click()
    time.sleep(2)

    # ✅ 추가 세부기록 크롤링
    third_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not third_all:
            third_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        third_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        third_all.extend(third_rows)
    df_third = pd.DataFrame(third_all, columns=third_cols)

    # ✅ 필요한 열만 추출
    stats_to_add = ['XBH', 'GO', 'AO', 'GO/AO', 'GW RBI', 'BB/K', 'P/PA', 'ISOP', 'XR', 'GPA']
    cols_available = [col for col in stats_to_add if col in df_third.columns]
    df_stats = df_third[['선수명', '팀명'] + cols_available]

    # ✅ 병합 후 시즌 추가
    df_final = pd.merge(df_final, df_stats, on=['선수명', '팀명'], how='left')
    df_final["시즌"] = year

    all_seasons.append(df_final)
    driver.quit()

# ✅ 전체 데이터 합치기
df_all = pd.concat(all_seasons, ignore_index=True)
df_all.to_csv("키움_타자_2025(최신데이터).csv", index=False, encoding="utf-8-sig")
print(df_all.head())


  순위  선수명  팀명    AVG    G   PA   AB   R    H  2B  ...   GO   AO GO/AO GW RBI  \
0  1  이재상  키움  0.500    2    2    2   0    1   0  ...    0    0     -      0   
1  2  송성문  키움  0.316  121  540  478  82  151  30  ...  121  134  0.90      8   
2  3  이주형  키움  0.313   11   17   16   2    5   0  ...    2    2  1.00      0   
3  3  이주형  키움  0.313   11   17   16   2    5   0  ...  111   73  1.52      4   
4  3  이주형  키움  0.313  106  434  374  45   93  15  ...    2    2  1.00      0   

   BB/K  P/PA   ISOP    XR    GPA    시즌  
0  0.00  3.00  0.000   0.4  0.350  2025  
1  0.77  3.74  0.215  96.4  0.308  2025  
2  0.14  3.88  0.000   1.8  0.237  2025  
3  0.34  3.78  0.126  52.9  0.250  2025  
4  0.14  3.88  0.000   1.8  0.237  2025  

[5 rows x 38 columns]


### 투수 데이터

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# ✅ 전체 데이터를 저장할 리스트
all_seasons = []

# ✅ 연도별 반복
for year in range(2018, 2026):
    driver = webdriver.Chrome()
    url = "https://www.koreabaseball.com/Record/Player/PitcherBasic/Basic1.aspx"
    driver.get(url)

    # ✅ 연도/팀 선택
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlSeason_ddlSeason")).select_by_visible_text(str(year))
    time.sleep(1)

    # ✅ 팀명: 2021년부터는 SSG, 이전은 SK
    team_name = "NC" if year >= 2019 else "NC"
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlTeam_ddlTeam")).select_by_visible_text(team_name)
    time.sleep(2)


    # ✅ 페이지 수 확인 함수
    def get_total_pages():
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pages = soup.select('div.paging a')
        numbers = [int(p.text.strip()) for p in pages if p.text.strip().isdigit()]
        return max(numbers) if numbers else 1

    # ✅ 기본기록 크롤링
    basic_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not basic_all:
            basic_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        basic_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        basic_all.extend(basic_rows)
    df_basic = pd.DataFrame(basic_all, columns=basic_cols)

    # ✅ 세부기록 탭 클릭
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[2]/a[2]').click()
    time.sleep(2)

    # ✅ 세부기록 크롤링
    detail_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not detail_all:
            detail_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        detail_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        detail_all.extend(detail_rows)
    df_detail = pd.DataFrame(detail_all, columns=detail_cols)

    # ✅ 기본 + 세부기록 병합
    df_basic = df_basic.drop(columns=['순위', 'AVG'], errors='ignore')
    df_merged = pd.merge(df_detail, df_basic, on=['선수명', '팀명'], how='left')

    # ✅ PH-BA까지 정렬
    pre_stats = [
        '순위', '선수명', '팀명', 'ERA', 'G', 'W', 'L', 'SV', 'HLD',
        'WPCT', 'IP', 'H', 'HR', 'BB', 'HBP', 'SO', 'R', 'ER', 'WHIP', 'CG', 'SHO',
        'QS', 'BSV', 'TBF', 'NP', 'AVG', '2B', '3B', 'SAC', 'SF', 'IBB', 'WP', 'BK'
    ]
    pre_stats = [col for col in pre_stats if col in df_merged.columns]
    df_final = df_merged[pre_stats]

    # ✅ XBH ~ GPA 탭 클릭
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[1]/ul/li[2]/a').click()
    time.sleep(2)

    # ✅ 추가 세부기록 크롤링
    third_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not third_all:
            third_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        third_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        third_all.extend(third_rows)
    df_third = pd.DataFrame(third_all, columns=third_cols)

    # ✅ 필요한 열만 추출
    stats_to_add = ['GS', 'Wgs', 'Wgr', 'GF', 'SVO', 'TS', 'GDP', 'GO', 'AO', 'GO/AO']
    cols_available = [col for col in stats_to_add if col in df_third.columns]
    df_stats = df_third[['선수명', '팀명'] + cols_available]

    # ✅ 병합 후 시즌 추가
    df_final = pd.merge(df_final, df_stats, on=['선수명', '팀명'], how='left')
    df_final["시즌"] = year

    all_seasons.append(df_final)
    driver.quit()

# ✅ 전체 데이터 합치기
df_all = pd.concat(all_seasons, ignore_index=True)
df_all.to_csv("NC_투수_2018_2025.csv", index=False, encoding="utf-8-sig")
print(df_all.head())

  순위   선수명  팀명   G  W   L SV HLD   WPCT       IP  ... Wgs Wgr GF SVO TS GDP  \
0  1   심규범  NC   1  0   0  0   0      -      2/3  ...   0   0  1   0  0   0   
1  1   이우석  NC   1  0   0  0   0      -      1/3  ...   0   0  1   0  0   0   
2  3   김재균  NC   4  0   0  0   0      -    5 2/3  ...   0   0  2   0  0   0   
3  4   박진우  NC  11  1   0  0   1  1.000   19 2/3  ...   0   1  1   2  0   3   
4  5  왕웨이중  NC  25  7  10  0   0  0.412  141 2/3  ...   7   0  0   0  0  19   

    GO   AO GO/AO    시즌  
0    1    1  1.00  2018  
1    0    0     -  2018  
2    7    6  1.17  2018  
3   25   14  1.79  2018  
4  165  126  1.31  2018  

[5 rows x 43 columns]


### 수비 데이터

In [36]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# ✅ 연도별 데이터를 저장할 리스트
all_defense = []

# ✅ 연도별 반복
for year in range(2025, 2026):
    print(f"[{year} 시즌 시작]")
    driver = webdriver.Chrome()
    url = "https://www.koreabaseball.com/Record/Player/Defense/Basic.aspx"
    driver.get(url)

    # ✅ 연도 선택
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlSeason_ddlSeason")).select_by_visible_text(str(year))
    time.sleep(1)

    # ✅ 팀 선택: 2021년부터는 SSG, 이전은 SK
    team_name = "키움" if year >= 2019 else "키움"
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlTeam_ddlTeam")).select_by_visible_text(team_name)
    time.sleep(2)

    # ✅ 페이지 수 확인 함수
    def get_total_pages():
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pages = soup.select('div.paging a')
        numbers = [int(p.text.strip()) for p in pages if p.text.strip().isdigit()]
        return max(numbers) if numbers else 1

    # ✅ 페이지 순회하며 데이터 수집
    defense_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/table/tbody'))
    )
    total_pages = get_total_pages()

    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}'))
                )
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(2)
            except Exception as e:
                print(f"[{year} 페이지 {page}] 클릭 실패: {e}")
                break

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not defense_all:
            defense_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        defense_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        defense_all.extend(defense_rows)

    # ✅ 데이터프레임화 + 시즌 열 추가
    df_defense = pd.DataFrame(defense_all, columns=defense_cols)
    df_defense['시즌'] = year
    all_defense.append(df_defense)

    print(f"[{year} 시즌 완료]\n")
    driver.quit()

# ✅ 전체 병합 후 저장
df_all_defense = pd.concat(all_defense, ignore_index=True)
df_all_defense.to_csv("키움_수비_2025(최신데이터).csv", index=False, encoding="utf-8-sig")
print("✅ 수비 데이터 저장 완료")

[2025 시즌 시작]
[2025 시즌 완료]

✅ 수비 데이터 저장 완료


### 주루 데이터

In [46]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# ✅ 연도별 데이터를 저장할 리스트
all_defense = []

# ✅ 연도별 반복
for year in range(2025, 2026):
    print(f"[{year} 시즌 시작]")
    driver = webdriver.Chrome()
    url = "https://www.koreabaseball.com/Record/Player/Runner/Basic.aspx"
    driver.get(url)

    # ✅ 연도 선택
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlSeason_ddlSeason")).select_by_visible_text(str(year))
    time.sleep(1)

    # ✅ 팀 선택: 2021년부터는 SSG, 이전은 SK
    team_name = "키움" if year >= 2019 else "키움"
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlTeam_ddlTeam")).select_by_visible_text(team_name)
    time.sleep(2)

    # ✅ 페이지 수 확인 함수
    def get_total_pages():
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pages = soup.select('div.paging a')
        numbers = [int(p.text.strip()) for p in pages if p.text.strip().isdigit()]
        return max(numbers) if numbers else 1

    # ✅ 페이지 순회하며 데이터 수집
    defense_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/table/tbody'))
    )
    total_pages = get_total_pages()

    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}'))
                )
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(2)
            except Exception as e:
                print(f"[{year} 페이지 {page}] 클릭 실패: {e}")
                break

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not defense_all:
            defense_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        defense_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        defense_all.extend(defense_rows)

    # ✅ 데이터프레임화 + 시즌 열 추가
    df_defense = pd.DataFrame(defense_all, columns=defense_cols)
    df_defense['시즌'] = year
    all_defense.append(df_defense)

    print(f"[{year} 시즌 완료]\n")
    driver.quit()

# ✅ 전체 병합 후 저장
df_all_defense = pd.concat(all_defense, ignore_index=True)
df_all_defense.to_csv("키움_주루_2025(최신데이터).csv", index=False, encoding="utf-8-sig")
print("✅ 수비 데이터 저장 완료")

[2025 시즌 시작]
[2025 시즌 완료]

✅ 수비 데이터 저장 완료


### 투수 수정 데이터

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

# ✅ 전체 데이터를 저장할 리스트
all_seasons = []

# ✅ 연도별 반복
for year in range(2018, 2026):
    driver = webdriver.Chrome()
    url = "https://www.koreabaseball.com/Record/Player/PitcherBasic/Basic1.aspx"
    driver.get(url)

    # ✅ 연도/팀 선택
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlSeason_ddlSeason")).select_by_visible_text(str(year))
    time.sleep(1)
    team_name = "키움" if year >= 2019 else "키움"  # 변경 가능
    Select(driver.find_element(By.ID, "cphContents_cphContents_cphContents_ddlTeam_ddlTeam")).select_by_visible_text(team_name)
    time.sleep(2)

    # ✅ 페이지 수 확인 함수
    def get_total_pages():
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        pages = soup.select('div.paging a')
        numbers = [int(p.text.strip()) for p in pages if p.text.strip().isdigit()]
        return max(numbers) if numbers else 1

    # ✅ 기본기록
    basic_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not basic_all:
            basic_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        basic_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        basic_all.extend(basic_rows)
    df_basic = pd.DataFrame(basic_all, columns=basic_cols)

    # ✅ 세부기록 탭 클릭
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[2]/a[2]').click()
    time.sleep(2)

    # ✅ 세부기록
    detail_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not detail_all:
            detail_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        detail_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        detail_all.extend(detail_rows)
    df_detail = pd.DataFrame(detail_all, columns=detail_cols)

    # ✅ 기본 + 세부기록 병합
    df_merged = pd.merge(df_detail, df_basic.drop(columns=['순위'], errors='ignore'), on=['선수명', '팀명'], how='left')

    # ✅ 추가 세부기록 탭 클릭 (GO/AO 등)
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[1]/ul/li[2]/a').click()
    time.sleep(2)

    third_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not third_all:
            third_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        third_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        third_all.extend(third_rows)
    df_third = pd.DataFrame(third_all, columns=third_cols)
    stats_to_add = ['GS', 'Wgs', 'Wgr', 'GF', 'SVO', 'TS', 'GDP', 'GO', 'AO', 'GO/AO']
    df_stats = df_third[['선수명', '팀명'] + [col for col in stats_to_add if col in df_third.columns]]
    df_merged = pd.merge(df_merged, df_stats, on=['선수명', '팀명'], how='left')

    # ✅ OPS 탭 클릭 (네 번째 탭)
    driver.find_element(By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[2]/div[2]/a[2]').click()
    time.sleep(2)

    ops_all = []
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//*[@id="cphContents_cphContents_cphContents_udpContent"]/div[3]/table/tbody'))
    )
    total_pages = get_total_pages()
    for page in range(1, total_pages + 1):
        if page != 1:
            try:
                btn = driver.find_element(By.ID, f'cphContents_cphContents_cphContents_ucPager_btnNo{page}')
                driver.execute_script("arguments[0].click();", btn)
                time.sleep(1.5)
            except:
                break
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        thead = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > thead')
        tbody = soup.select_one('#cphContents_cphContents_cphContents_udpContent > div.record_result > table > tbody')
        if not ops_all:
            ops_cols = [th.get_text(strip=True) for th in thead.find_all('th')]
        ops_rows = [[td.get_text(strip=True) for td in tr.find_all('td')] for tr in tbody.find_all('tr')]
        ops_all.extend(ops_rows)
    df_ops = pd.DataFrame(ops_all, columns=ops_cols)
    ops_stats = ['BABIP', 'P/G', 'P/IP', 'K/9', 'BB/9', 'K/BB', 'OBP', 'SLG', 'OPS']
    df_ops = df_ops[['선수명', '팀명'] + [col for col in ops_stats if col in df_ops.columns]]
    df_merged = pd.merge(df_merged, df_ops, on=['선수명', '팀명'], how='left')

    # ✅ 시즌 추가
    df_merged["시즌"] = year
    all_seasons.append(df_merged)
    driver.quit()

# ✅ 전체 병합 및 저장
df_all = pd.concat(all_seasons, ignore_index=True)
df_all.to_csv("키움_투수_2025(최신데이터).csv", index=False, encoding="utf-8-sig")
print(df_all.head())


  순위   선수명  팀명 ERA_x CG SHO QS BSV  TBF    NP  ...  BABIP   P/G  P/IP   K/9  \
0  1   김동규  키움  0.00  0   0  0   0    5    16  ...  0.250  16.0  16.0  0.00   
1  2   주승우  키움  2.45  0   0  0   4  179   713  ...  0.241  17.0  16.2  7.16   
2  3    웰스  키움  3.15  0   0  2   0   85   329  ...  0.290  82.3  16.5  7.20   
3  4  로젠버그  키움  3.23  0   0  6   0  320  1265  ...  0.286  97.3  16.8  9.56   
4  5  알칸타라  키움  3.58  0   0  9   0  342  1236  ...  0.321  95.1  14.9  7.05   

   BB/9  K/BB    OBP    SLG    OPS    시즌  
0  9.00  0.00  0.400  0.250  0.650  2025  
1  3.48  2.06  0.305  0.361  0.666  2025  
2  2.70  2.67  0.294  0.273  0.567  2025  
3  2.99  3.20  0.290  0.331  0.621  2025  
4  0.87  8.13  0.302  0.437  0.739  2025  

[5 rows x 54 columns]


### 데이터 크롤링 코드 (Statiz 사이트 크롤링)

In [1]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능 여부: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA를 사용할 수 없습니다.")
    

PyTorch 버전: 2.5.1+cu118
CUDA 사용 가능 여부: True
GPU 이름: NVIDIA GeForce RTX 4060 Ti


### Statiz + KBO 공식 정적 데이터 수집 코드

In [5]:
pip install pandas requests selenium webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
import os, time, random

# ==============================
# 0. 저장 경로 및 세션 설정
# ==============================
SAVE_DIR = r"C:\Users\USER\Desktop"
os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://statiz.sporki.com/",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8"
})

# ==============================
# 1. 안전 요청 함수 (403 방지)
# ==============================
def safe_request(url, max_retries=5):
    for attempt in range(max_retries):
        try:
            res = session.get(url, timeout=15)
            if res.status_code == 200:
                return res
            elif res.status_code == 403:
                print(f"[403 감지] {attempt+1}/{max_retries}회차 → 300초(5분) 대기 후 재시도")
                time.sleep(300)
        except Exception as e:
            print(f"[에러] {url}: {e}")
        time.sleep(random.uniform(8, 12))
    print(f"[실패] {url} → 요청 포기")
    return None

# ==============================
# 2. 연간 경기 ID 수집 (2025년 6월까지 제한)
# ==============================
def crawlAnnualGameIds(year):
    game_ids = []
    for month in range(3, 11):
        if year == 2025 and month > 6:  # ✅ 2025년 6월까지만
            continue

        url = f"https://statiz.sporki.com/schedule/?year={year}&month={month}"
        res = safe_request(url)
        if not res:
            continue
        soup = BeautifulSoup(res.text, "html.parser")

        cal = soup.find("div", class_="calendar_area")
        if not cal:
            continue

        for td in cal.find_all("td"):
            span_day = td.find("span", class_="day")
            div_game = td.find("div", class_="game_schedule_m")
            if span_day is None or div_game is None:
                continue

            # ✅ 2025년 6월 30일까지 제한
            if year == 2025 and month == 6 and int(span_day.text) > 30:
                continue

            for a in td.find_all("a"):
                if a.find("span", class_="weather rain") is None:
                    gid = re.search(r"s_no=(\d+)", a.get("href")).group(1)
                    game_ids.append(gid)
        time.sleep(random.uniform(8, 12))
    return game_ids

# ==============================
# 3. 박스스코어 데이터 추출 (박스스코어 없으면 스킵)
# ==============================
def extract_boxscore(game_id):
    url = f"https://statiz.sporki.com/schedule/?m=boxscore&s_no={game_id}"
    res = safe_request(url)
    if not res:
        return [], [], [], []

    soup = BeautifulSoup(res.text, "html.parser")
    div_schedule = soup.find("div", class_="game_schedule result")
    if div_schedule is None:
        print(f"[스킵] 박스스코어 없음 → {game_id}")
        return [], [], [], []

    div_score = div_schedule.find("div", class_="score")
    spans = div_score.find_all("span")
    date_txt = div_score.find("div", class_="txt").text.strip().split(", ")[1]
    game_date = f"{game_id[:4]}-{date_txt}"
    away_score, home_score = spans[0].text.strip(), spans[1].text.strip()

    tables = soup.find_all("div", class_="box_type_boared")
    if len(tables) < 4:  # ✅ 박스스코어가 4개 미만이면 스킵
        print(f"[스킵] 박스스코어 테이블 부족 → {game_id}")
        return [], [], [], []

    def extract_team_name(div_team_bat):
        name = div_team_bat.find("div", class_="box_head").text.strip()
        return re.search(r"\((.*?)\)", name).group(1)

    def extract_bat(div, game_id, game_date, team_type, team_name, score):
        rows = div.find_all("tr")
        individual, team_row = [], []
        for r in rows[1:-1]:
            cols = [c.text.strip() for c in r.find_all("td")[1:]]
            individual.append([game_id, game_date, team_type, team_name, score] + cols)
        team_row = [game_id, game_date, team_type, team_name, score] + \
                   [c.text.strip() for c in rows[-1].find_all("td")[1:]]
        return individual, team_row

    def extract_pitch(div, game_id, game_date, team_type, team_name, score):
        rows = div.find_all("tr")
        individual, team_row = [], []
        for r in rows[1:-1]:
            cols = [c.text.strip() for c in r.find_all("td")]
            individual.append([game_id, game_date, team_type, team_name, score] + cols)
        team_row = [game_id, game_date, team_type, team_name, score] + \
                   [c.text.strip() for c in rows[-1].find_all("td")[1:]]
        return individual, team_row

    away_bat_rows, away_team_bat_row = extract_bat(tables[0], game_id, game_date, "away", extract_team_name(tables[0]), away_score)
    home_bat_rows, home_team_bat_row = extract_bat(tables[1], game_id, game_date, "home", extract_team_name(tables[1]), home_score)
    away_pitch_rows, away_team_pitch_row = extract_pitch(tables[2], game_id, game_date, "away", extract_team_name(tables[0]), away_score)
    home_pitch_rows, home_team_pitch_row = extract_pitch(tables[3], game_id, game_date, "home", extract_team_name(tables[1]), home_score)

    return (
        away_bat_rows + home_bat_rows,
        away_pitch_rows + home_pitch_rows,
        [away_team_bat_row, home_team_bat_row],
        [away_team_pitch_row, home_team_pitch_row]
    )

# ==============================
# 4. 연도별 저장
# ==============================
def save_incremental(df, filename):
    path = os.path.join(SAVE_DIR, filename)
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")

# ==============================
# 5. 메인 실행 (슈퍼 슬로우 + 안전 스킵)
# ==============================
for year in range(2018, 2026):
    print(f"=== {year} 시즌 수집 시작 ===")
    game_ids = crawlAnnualGameIds(year)
    print(f"총 {len(game_ids)} 경기 예정")

    batting_all, pitching_all, team_bat_all, team_pitch_all = [], [], [], []

    for idx, gid in enumerate(game_ids):
        try:
            bat, pitch, t_bat, t_pitch = extract_boxscore(gid)
            batting_all.extend(bat)
            pitching_all.extend(pitch)
            team_bat_all.extend(t_bat)
            team_pitch_all.extend(t_pitch)
        except Exception as e:
            print(f"[에러] {gid} → {e}")

        time.sleep(random.uniform(8, 12))  # ✅ 초안전
        if (idx + 1) % 10 == 0:
            print(f"💤 {idx+1}경기 진행 → 2~3분 휴식")
            time.sleep(random.uniform(120, 180))

    save_incremental(pd.DataFrame(batting_all), f"batting_{year}.csv")
    save_incremental(pd.DataFrame(pitching_all), f"pitching_{year}.csv")
    save_incremental(pd.DataFrame(team_bat_all), f"team_bat_{year}.csv")
    save_incremental(pd.DataFrame(team_pitch_all), f"team_pitch_{year}.csv")

    print(f"=== {year} 시즌 저장 완료 ===")
    if year != 2025:  # ✅ 마지막 시즌 후 휴식 불필요
        print("💤 10~15분 휴식 (다음 시즌 대비)")
        time.sleep(random.uniform(600, 900))

print("✅ 모든 시즌 수집 완료!")


=== 2018 시즌 수집 시작 ===
총 729 경기 예정
💤 10경기 진행 → 2~3분 휴식
💤 20경기 진행 → 2~3분 휴식
💤 30경기 진행 → 2~3분 휴식
💤 40경기 진행 → 2~3분 휴식
💤 50경기 진행 → 2~3분 휴식
💤 60경기 진행 → 2~3분 휴식
💤 70경기 진행 → 2~3분 휴식
💤 80경기 진행 → 2~3분 휴식
💤 90경기 진행 → 2~3분 휴식
💤 100경기 진행 → 2~3분 휴식
💤 110경기 진행 → 2~3분 휴식
💤 120경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180139
💤 130경기 진행 → 2~3분 휴식
💤 140경기 진행 → 2~3분 휴식
💤 150경기 진행 → 2~3분 휴식
💤 160경기 진행 → 2~3분 휴식
💤 170경기 진행 → 2~3분 휴식
💤 180경기 진행 → 2~3분 휴식
💤 190경기 진행 → 2~3분 휴식
💤 200경기 진행 → 2~3분 휴식
💤 210경기 진행 → 2~3분 휴식
💤 220경기 진행 → 2~3분 휴식
💤 230경기 진행 → 2~3분 휴식
💤 240경기 진행 → 2~3분 휴식
💤 250경기 진행 → 2~3분 휴식
💤 260경기 진행 → 2~3분 휴식
💤 270경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180305
💤 280경기 진행 → 2~3분 휴식
💤 290경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180314
💤 300경기 진행 → 2~3분 휴식
💤 310경기 진행 → 2~3분 휴식
💤 320경기 진행 → 2~3분 휴식
💤 330경기 진행 → 2~3분 휴식
💤 340경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180370
💤 350경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180382
💤 360경기 진행 → 2~3분 휴식
💤 370경기 진행 → 2~3분 휴식
[스킵] 박스스코어 테이블 부족 → 20180403
💤 380경기 진행 → 2~3분 휴식
💤 3

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime
import os, time, random

# ==============================
# 0. 저장 경로 및 세션 설정
# ==============================
SAVE_DIR = r"C:\Users\USER\Desktop"
os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Referer": "https://statiz.sporki.com/",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8"
})

# ==============================
# 1. 안전 요청 함수 (403 방지)
# ==============================
def safe_request(url, max_retries=5):
    for attempt in range(max_retries):
        try:
            res = session.get(url, timeout=15)
            if res.status_code == 200:
                return res
            elif res.status_code == 403:
                print(f"[403 감지] {attempt+1}/{max_retries}회차 → 300초(5분) 대기 후 재시도")
                time.sleep(300)
        except Exception as e:
            print(f"[에러] {url}: {e}")
        time.sleep(random.uniform(8, 12))
    print(f"[실패] {url} → 요청 포기")
    return None

# ==============================
# 2. 연간 경기 ID 수집 (2025년 6월까지 제한)
# ==============================
def crawlAnnualGameIds(year):
    game_ids = []
    for month in range(3, 11):
        if year == 2025 and month > 6:  # ✅ 2025년 6월까지만
            continue

        url = f"https://statiz.sporki.com/schedule/?year={year}&month={month}"
        res = safe_request(url)
        if not res:
            continue
        soup = BeautifulSoup(res.text, "html.parser")

        cal = soup.find("div", class_="calendar_area")
        if not cal:
            continue

        for td in cal.find_all("td"):
            span_day = td.find("span", class_="day")
            div_game = td.find("div", class_="game_schedule_m")
            if span_day is None or div_game is None:
                continue

            # ✅ 2025년 6월 30일까지 제한
            if year == 2025 and month == 6 and int(span_day.text) > 30:
                continue

            for a in td.find_all("a"):
                if a.find("span", class_="weather rain") is None:
                    gid = re.search(r"s_no=(\d+)", a.get("href")).group(1)
                    game_ids.append(gid)
        time.sleep(random.uniform(8, 12))
    return game_ids

# ==============================
# 3. 박스스코어 데이터 추출 (박스스코어 없으면 스킵)
# ==============================
def extract_boxscore(game_id):
    url = f"https://statiz.sporki.com/schedule/?m=boxscore&s_no={game_id}"
    res = safe_request(url)
    if not res:
        return [], [], [], []

    soup = BeautifulSoup(res.text, "html.parser")
    div_schedule = soup.find("div", class_="game_schedule result")
    if div_schedule is None:
        print(f"[스킵] 박스스코어 없음 → {game_id}")
        return [], [], [], []

    div_score = div_schedule.find("div", class_="score")
    spans = div_score.find_all("span")
    date_txt = div_score.find("div", class_="txt").text.strip().split(", ")[1]
    game_date = f"{game_id[:4]}-{date_txt}"
    away_score, home_score = spans[0].text.strip(), spans[1].text.strip()

    tables = soup.find_all("div", class_="box_type_boared")
    if len(tables) < 4:  # ✅ 박스스코어가 4개 미만이면 스킵
        print(f"[스킵] 박스스코어 테이블 부족 → {game_id}")
        return [], [], [], []

    def extract_team_name(div_team_bat):
        name = div_team_bat.find("div", class_="box_head").text.strip()
        return re.search(r"\((.*?)\)", name).group(1)

    def extract_bat(div, game_id, game_date, team_type, team_name, score):
        rows = div.find_all("tr")
        individual, team_row = [], []
        for r in rows[1:-1]:
            cols = [c.text.strip() for c in r.find_all("td")[1:]]
            individual.append([game_id, game_date, team_type, team_name, score] + cols)
        team_row = [game_id, game_date, team_type, team_name, score] + \
                   [c.text.strip() for c in rows[-1].find_all("td")[1:]]
        return individual, team_row

    def extract_pitch(div, game_id, game_date, team_type, team_name, score):
        rows = div.find_all("tr")
        individual, team_row = [], []
        for r in rows[1:-1]:
            cols = [c.text.strip() for c in r.find_all("td")]
            individual.append([game_id, game_date, team_type, team_name, score] + cols)
        team_row = [game_id, game_date, team_type, team_name, score] + \
                   [c.text.strip() for c in rows[-1].find_all("td")[1:]]
        return individual, team_row

    away_bat_rows, away_team_bat_row = extract_bat(tables[0], game_id, game_date, "away", extract_team_name(tables[0]), away_score)
    home_bat_rows, home_team_bat_row = extract_bat(tables[1], game_id, game_date, "home", extract_team_name(tables[1]), home_score)
    away_pitch_rows, away_team_pitch_row = extract_pitch(tables[2], game_id, game_date, "away", extract_team_name(tables[0]), away_score)
    home_pitch_rows, home_team_pitch_row = extract_pitch(tables[3], game_id, game_date, "home", extract_team_name(tables[1]), home_score)

    return (
        away_bat_rows + home_bat_rows,
        away_pitch_rows + home_pitch_rows,
        [away_team_bat_row, home_team_bat_row],
        [away_team_pitch_row, home_team_pitch_row]
    )

# ==============================
# 4. 연도별 저장
# ==============================
def save_incremental(df, filename):
    path = os.path.join(SAVE_DIR, filename)
    if os.path.exists(path):
        df.to_csv(path, mode="a", header=False, index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")

# ==============================
# 5. 메인 실행 (슈퍼 슬로우 + 안전 스킵)
# ==============================
for year in range(2024, 2026):
    print(f"=== {year} 시즌 수집 시작 ===")
    game_ids = crawlAnnualGameIds(year)
    print(f"총 {len(game_ids)} 경기 예정")

    batting_all, pitching_all, team_bat_all, team_pitch_all = [], [], [], []

    for idx, gid in enumerate(game_ids):
        try:
            bat, pitch, t_bat, t_pitch = extract_boxscore(gid)
            batting_all.extend(bat)
            pitching_all.extend(pitch)
            team_bat_all.extend(t_bat)
            team_pitch_all.extend(t_pitch)
        except Exception as e:
            print(f"[에러] {gid} → {e}")

        time.sleep(random.uniform(8, 12))  # ✅ 초안전
        if (idx + 1) % 10 == 0:
            print(f"💤 {idx+1}경기 진행 → 2~3분 휴식")
            time.sleep(random.uniform(120, 180))

    save_incremental(pd.DataFrame(batting_all), f"batting_{year}.csv")
    save_incremental(pd.DataFrame(pitching_all), f"pitching_{year}.csv")
    save_incremental(pd.DataFrame(team_bat_all), f"team_bat_{year}.csv")
    save_incremental(pd.DataFrame(team_pitch_all), f"team_pitch_{year}.csv")

    print(f"=== {year} 시즌 저장 완료 ===")
    if year != 2025:  # ✅ 마지막 시즌 후 휴식 불필요
        print("💤 10~15분 휴식 (다음 시즌 대비)")
        time.sleep(random.uniform(600, 900))

print("✅ 모든 시즌 수집 완료!")


=== 2024 시즌 수집 시작 ===
총 736 경기 예정
💤 10경기 진행 → 2~3분 휴식
💤 20경기 진행 → 2~3분 휴식
💤 30경기 진행 → 2~3분 휴식
💤 40경기 진행 → 2~3분 휴식
💤 50경기 진행 → 2~3분 휴식
💤 60경기 진행 → 2~3분 휴식
💤 70경기 진행 → 2~3분 휴식
💤 80경기 진행 → 2~3분 휴식
💤 90경기 진행 → 2~3분 휴식
💤 100경기 진행 → 2~3분 휴식
💤 110경기 진행 → 2~3분 휴식
💤 120경기 진행 → 2~3분 휴식
💤 130경기 진행 → 2~3분 휴식
💤 140경기 진행 → 2~3분 휴식
💤 150경기 진행 → 2~3분 휴식
💤 160경기 진행 → 2~3분 휴식
💤 170경기 진행 → 2~3분 휴식
💤 180경기 진행 → 2~3분 휴식
💤 190경기 진행 → 2~3분 휴식
💤 200경기 진행 → 2~3분 휴식
💤 210경기 진행 → 2~3분 휴식
💤 220경기 진행 → 2~3분 휴식
💤 230경기 진행 → 2~3분 휴식
💤 240경기 진행 → 2~3분 휴식
💤 250경기 진행 → 2~3분 휴식
💤 260경기 진행 → 2~3분 휴식
💤 270경기 진행 → 2~3분 휴식
💤 280경기 진행 → 2~3분 휴식
💤 290경기 진행 → 2~3분 휴식
💤 300경기 진행 → 2~3분 휴식
💤 310경기 진행 → 2~3분 휴식
💤 320경기 진행 → 2~3분 휴식
💤 330경기 진행 → 2~3분 휴식
💤 340경기 진행 → 2~3분 휴식
💤 350경기 진행 → 2~3분 휴식
💤 360경기 진행 → 2~3분 휴식
💤 370경기 진행 → 2~3분 휴식
💤 380경기 진행 → 2~3분 휴식
💤 390경기 진행 → 2~3분 휴식
💤 400경기 진행 → 2~3분 휴식
💤 410경기 진행 → 2~3분 휴식
💤 420경기 진행 → 2~3분 휴식
💤 430경기 진행 → 2~3분 휴식
💤 440경기 진행 → 2~3분 휴식
💤 450경기 진행 → 2~3분 휴식
💤 460경기 진행 → 2~3분 휴식
💤 470경기 진

### KBO 박스스코어 동적 데이터 크롤링 (selenium 기반)

기본 데이터 수집

In [6]:
pip install selenium pandas


Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install selenium pandas chromedriver-autoinstaller


Note: you may need to restart the kernel to use updated packages.


### 1단계: 경기 로그 수집

In [4]:
pip install webdriver-manager


Note: you may need to restart the kernel to use updated packages.


In [12]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
import os

SAVE_DIR = r"C:\Users\THKIM\Desktop\3학년 여름방학 프로젝트\2025 KBO (한국프로야구) 오픈데이터 활용 공모전\data"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://sports.news.naver.com/"
}

def save_csv(df, filename):
    os.makedirs(SAVE_DIR, exist_ok=True)
    path = os.path.join(SAVE_DIR, filename)
    if os.path.exists(path):
        df.to_csv(path, mode='a', header=False, index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")

def safe_request(url, max_retries=3, delay=5):
    """차단 방지용 재시도 로직"""
    for attempt in range(max_retries):
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 200:
                return res
        except Exception as e:
            print(f"[경고] 요청 실패({attempt+1}/{max_retries}): {url} / {e}")
        time.sleep(delay)
    return None

def get_game_ids(date_str):
    url = f"https://sports.news.naver.com/kbaseball/schedule/index?date={date_str}"
    res = safe_request(url)
    if not res:
        return []
    soup = BeautifulSoup(res.text, "html.parser")
    games = []
    for a in soup.select(".hmb_list li a"):
        gid = a.get("href", "").split("gameCenter/")[1].split("?")[0] if "gameCenter" in a.get("href", "") else ""
        if gid:
            games.append(gid)
    return games

def get_boxscore(game_id, date_str):
    url = f"https://sports.news.naver.com/gameCenter/gameRecord?gameId={game_id}"
    res = safe_request(url)
    if not res:
        return [], []
    data = res.json()
    batting, pitching = [], []

    for team in ["homeTeamRecord", "awayTeamRecord"]:
        for b in data.get(team, {}).get("batters", []):
            b.update({"date": date_str, "game_id": game_id, "team": data.get(team, {}).get("teamName", "")})
            batting.append(b)
        for p in data.get(team, {}).get("pitchers", []):
            p.update({"date": date_str, "game_id": game_id, "team": data.get(team, {}).get("teamName", "")})
            pitching.append(p)
    return batting, pitching

def get_playbyplay(game_id, date_str):
    url = f"https://sports.news.naver.com/gameCenter/textRelay?gameId={game_id}"
    res = safe_request(url)
    if not res:
        return []
    data = res.json()
    playbyplay = []
    for inning in data.get("result", {}).get("relayText", []):
        for play in inning.get("relayTexts", []):
            playbyplay.append({
                "date": date_str,
                "game_id": game_id,
                "inning": inning.get("inning", ""),
                "batter": play.get("playerName", ""),
                "result": play.get("text", "")
            })
    return playbyplay

# ===== 메인 실행 =====
for year in range(2018, 2026):
    batting_list, pitching_list, playbyplay_list = [], [], []
    dates = pd.date_range(f"{year}-03-01", f"{year}-11-30")
    print(f"=== {year} 시즌 수집 시작 ===")

    for date in tqdm(dates):
        date_str = date.strftime("%Y-%m-%d")
        game_ids = get_game_ids(date_str)
        if not game_ids:
            continue

        for gid in game_ids:
            b, p = get_boxscore(gid, date_str)
            pbp = get_playbyplay(gid, date_str)
            batting_list.extend(b)
            pitching_list.extend(p)
            playbyplay_list.extend(pbp)
            time.sleep(1)  # 요청 간격 늘림(중요)

    if batting_list:
        save_csv(pd.DataFrame(batting_list), f"batting_{year}.csv")
    if pitching_list:
        save_csv(pd.DataFrame(pitching_list), f"pitching_{year}.csv")
    if playbyplay_list:
        save_csv(pd.DataFrame(playbyplay_list), f"playbyplay_{year}.csv")

    print(f"=== {year} 시즌 저장 완료 ===")

print("✅ 모든 시즌 수집 완료!")


=== 2018 시즌 수집 시작 ===


100%|██████████| 275/275 [00:21<00:00, 13.04it/s]


=== 2018 시즌 저장 완료 ===
=== 2019 시즌 수집 시작 ===


  5%|▌         | 15/275 [00:00<00:12, 20.62it/s]

[경고] 요청 실패(1/3): https://sports.news.naver.com/kbaseball/schedule/index?date=2019-03-18 / ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


...

=== 2023 시즌 저장 완료 ===
=== 2024 시즌 수집 시작 ===


100%|██████████| 275/275 [00:18<00:00, 14.87it/s]


=== 2024 시즌 저장 완료 ===
=== 2025 시즌 수집 시작 ===


100%|██████████| 275/275 [00:18<00:00, 15.01it/s]

=== 2025 시즌 저장 완료 ===
✅ 모든 시즌 수집 완료!
